# Black-box top-k sources and robust targets

Generate target→top sources and source→top robust targets using the shared transfer-view configuration.

# Black-box transfer analysis configuration

All three notebooks use the same `TransferViewConfig` object, so the attack matrix, source--target scatter, and top-k tables are directly comparable.

Key knobs:

- `source_grain` / `target_grain`: `family`, `architecture`, or `variant`.
- `comparison_mode`:
  - `inter_variant`: exclude exact same variant only.
  - `inter_architecture`: exclude same architecture.
  - `inter_model`: exclude same family.
- `variant_aggregate_mode`: how variants are collapsed into architectures/families: `mean`, `median`, `best_damage`, `best_robust`.
- `architecture_aggregate_mode`: how architectures are collapsed into families: `mean`, `median`, `best_damage`, `best_robust`.
- `split_ansatz_as_architecture`: when `True`, VQM/HVQL/etc. ansatz variants become architecture labels such as `VQM-Strong` or `HVQL-Random`.
- `blacklist_contains` and `model_blacklist` apply to all outputs.


In [1]:

from pathlib import Path
import sys

try:
    from analytics.paths import find_project_root, output_dir
except ModuleNotFoundError:
    # Allows running the notebook from a copied location.
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root / "src").exists():
        repo_root = repo_root.parent
    sys.path.insert(0, str(repo_root / "src"))
    from analytics.paths import find_project_root, output_dir

from analytics.attacks.transfer_role_tools import (
    TransferViewConfig,
    VIEW_TEMPLATES,
    describe_config,
    get_template,
    list_templates,
    load_or_build_pair_metrics,
    build_attack_matrix,
    build_source_target_scatter,
    build_top_sources_by_target,
    build_top_targets_by_source,
    plot_attack_matrix,
    plot_source_target_scatter,
    save_figure,
    write_outputs,
)

ROOT = find_project_root()
PAIR_METRICS_CACHE = output_dir(ROOT, "attack_transfer_role_tables") / "pair_metrics.csv"
OUT_DIR = output_dir(ROOT, "blackbox_transfer_views")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Template shortcuts:
# list_templates()
TEMPLATE_NAME = "architecture_mean"

CONFIG = get_template(
    TEMPLATE_NAME,
    source_grain="architecture",
    target_grain="architecture",
    comparison_mode="inter_architecture",
    metric="Transfer ASR",
    top_k=3,
    variant_aggregate_mode="mean",
    architecture_aggregate_mode="mean",
    split_ansatz_as_architecture=False,
    # Exact model names to exclude.
    model_blacklist=tuple(),
    # Substring rules, useful for quick exclusions such as ("HQuanv",).
    blacklist_contains=tuple(),
)

display(describe_config(CONFIG))


,value
source_grain,architecture
target_grain,architecture
comparison_mode,inter_architecture
metric,Transfer ASR
top_k,3
variant_aggregate_mode,mean
architecture_aggregate_mode,mean
split_ansatz_as_architecture,False
include_no_entangling_as_ansatz,True
model_blacklist,()


In [6]:
display(list_templates())

,template,source_grain,target_grain,comparison_mode,metric,top_k,variant_aggregate_mode,architecture_aggregate_mode,split_ansatz_as_architecture,include_no_entangling_as_ansatz,model_blacklist,blacklist_contains,source_family_filter,target_family_filter,source_architecture_filter,target_architecture_filter,source_contains,target_contains
0,variant_full,variant,variant,inter_variant,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
1,architecture_mean,architecture,architecture,inter_architecture,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
2,architecture_best_damage,architecture,architecture,inter_architecture,Transfer ASR,3,best_damage,mean,False,True,(),(),None,None,None,None,None,None
3,architecture_best_robust,architecture,architecture,inter_architecture,Transfer ASR,3,best_robust,mean,False,True,(),(),None,None,None,None,None,None
4,architecture_ansatz_mean,architecture,architecture,inter_architecture,Transfer ASR,3,mean,mean,True,True,(),(),None,None,None,None,None,None
5,family_mean_architecture_mean_variant,family,family,inter_variant,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
6,family_mean_architecture_best_damage_variant,family,family,inter_variant,Transfer ASR,3,best_damage,mean,False,True,(),(),None,None,None,None,None,None
7,family_mean_architecture_best_robust_variant,family,family,inter_variant,Transfer ASR,3,best_robust,mean,False,True,(),(),None,None,None,None,None,None
8,inter_family_architecture_mean,architecture,architecture,inter_model,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
9,top_variants_vs_target_architecture,variant,architecture,inter_architecture,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None


In [2]:

pair_df = load_or_build_pair_metrics(
    pair_metrics_cache=PAIR_METRICS_CACHE,
    rebuild=False,
)
print(f"Loaded pair metrics: {pair_df.shape[0]:,} rows × {pair_df.shape[1]:,} columns")


Loaded pair metrics: 1,089 rows × 24 columns


In [3]:

# Most damaging sources for each target.
top_sources_wide, top_sources_long = build_top_sources_by_target(
    pair_df,
    CONFIG,
    rank_mode="damage",
)
print("Top damaging sources by target")
display(top_sources_wide)


Top damaging sources by target


,target,top 1 source (Transfer ASR),top 2 source (Transfer ASR),top 3 source (Transfer ASR)
0,Classical/CNN,Hybrid/HQuanv (80.7%),Classical/MLP (78.1%),Classical/ResNet18 (67.8%)
1,Classical/MLP,Hybrid/HQuanv (86.4%),Classical/CNN (83.7%),Classical/ResNet18 (75.0%)
2,Classical/MLPMixer,Classical/CNN (79.2%),Hybrid/HQuanv (77.1%),Classical/MLP (75.6%)
3,Classical/ResNet18,Hybrid/HPVQL (76.3%),Hybrid/HVQL (76.1%),Classical/MLPMixer (76.1%)
4,Hybrid/HPVQL,Classical/CNN (70.9%),Hybrid/HQuanv (69.8%),Classical/ResNet18 (69.7%)
5,Hybrid/HQE,Hybrid/HQuanv (85.0%),Classical/CNN (80.6%),Classical/MLP (80.5%)
6,Hybrid/HQuanv,Classical/MLP (80.3%),Classical/CNN (79.4%),Quantum/VQM (66.2%)
7,Hybrid/HVQL,Hybrid/HPVQL (72.5%),Classical/CNN (72.2%),Hybrid/HQuanv (71.4%)
8,Hybrid/HVQM,Hybrid/HQuanv (72.6%),Classical/CNN (69.8%),Classical/MLP (69.3%)
9,Quantum/VQM,Classical/MLP (66.2%),Hybrid/HQuanv (66.1%),Classical/CNN (65.0%)


In [4]:

# Most robust / least damaged targets for each source.
top_targets_wide, top_targets_long = build_top_targets_by_source(
    pair_df,
    CONFIG,
    rank_mode="robust",
)
print("Top robust targets by source")
display(top_targets_wide)


Top robust targets by source


,source,top 1 robust target (Transfer ASR),top 2 robust target (Transfer ASR),top 3 robust target (Transfer ASR)
0,Classical/CNN,Quantum/VQM (65.0%),Hybrid/HVQM (69.8%),Hybrid/HPVQL (70.9%)
1,Classical/MLP,Quantum/VQM (66.2%),Hybrid/HPVQL (67.9%),Hybrid/HVQL (69.0%)
2,Classical/MLPMixer,Quantum/VQM (54.6%),Hybrid/HVQM (61.8%),Hybrid/HQuanv (63.1%)
3,Classical/ResNet18,Quantum/VQM (55.1%),Hybrid/HQuanv (62.9%),Hybrid/HVQM (65.8%)
4,Hybrid/HPVQL,Quantum/VQM (53.7%),Hybrid/HQuanv (61.1%),Hybrid/HVQM (62.6%)
5,Hybrid/HQE,Quantum/VQM (43.1%),Hybrid/HVQM (43.8%),Hybrid/HPVQL (44.6%)
6,Hybrid/HQuanv,Quantum/VQM (66.1%),Hybrid/HPVQL (69.8%),Hybrid/HVQL (71.4%)
7,Hybrid/HVQL,Quantum/VQM (52.8%),Hybrid/HQuanv (57.7%),Hybrid/HVQM (62.7%)
8,Hybrid/HVQM,Quantum/VQM (59.8%),Hybrid/HQuanv (61.0%),Hybrid/HQE (64.2%)
9,Quantum/VQM,Classical/CNN (64.7%),Hybrid/HQuanv (66.2%),Hybrid/HPVQL (66.2%)


In [5]:

prefix = f"topk_{CONFIG.source_grain}_to_{CONFIG.target_grain}_{CONFIG.comparison_mode}_{CONFIG.metric.replace(' ', '_')}"
outputs = write_outputs(
    OUT_DIR,
    prefix=prefix,
    top_sources_wide=top_sources_wide,
    top_sources_long=top_sources_long,
    top_targets_wide=top_targets_wide,
    top_targets_long=top_targets_long,
)
outputs


{'top_sources_wide': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/topk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_sources_wide.csv'),
 'top_sources_wide_latex': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/topk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_sources_wide.tex'),
 'top_sources_long': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/topk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_sources_long.csv'),
 'top_targets_wide': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/topk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_targets_wide.csv'),
 'top_targets_wide_latex': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/blackbox_transfer_views/topk_architecture_to_architecture_inter_architecture_Transfer_ASR_top_targets_wide.tex'),
 'top_targets_long': WindowsPath('C:/Users/fwilh/Desk